In [ ]:
import json
import os

from typing import Dict

import geopandas as gpd
import pandas as pd


def load_json(file_name):
    with open(file_name, 'r') as file:
        return json.load(file)


def load_and_merge_annotations(coco_labels_file: str, metadata_file: str, categories: Dict[int, str]) -> pd.DataFrame:
    coco_data = load_json(coco_labels_file)
    coco_data["annotations"] = [
        annot for annot in coco_data["annotations"] 
        if annot["category_id"] in categories.keys()
    ]

    images_df = pd.DataFrame(data=coco_data["images"]).set_index("id")[["file_name", "width", "height"]]
    images_df.insert(
        loc=1, 
        column="name", 
        value=[os.path.splitext(os.path.basename(file))[0] for file in images_df["file_name"]]
    )

    metadata_gdf = gpd.read_file(metadata_file)[["file_name", "group", "geometry"]]
    metadata_gdf["name"] = [os.path.splitext(file_name)[0] for file_name in metadata_gdf["file_name"]]
    metadata_gdf.set_index("name", inplace=True)
    metadata_gdf.drop(columns="file_name", inplace=True)

    annotations_df = pd.DataFrame(data=coco_data["annotations"]).set_index("id")

    annotations_df[["x_min", "y_min", "x_width", "y_height"]] = annotations_df["bbox"].apply(lambda row: pd.Series(data=row))
    annotations_df["area"] = annotations_df["x_width"] * annotations_df["y_height"]

    annotations_df = (
        annotations_df
        .join(images_df, on="image_id", how="left")
        .join(metadata_gdf, on="name", how="left")
    )

    return annotations_df


def count_categories(image_df: pd.DataFrame, categories: Dict[int, str]) -> pd.Series:
    counts = image_df["category_id"].value_counts()

    data = {
        categories[key]: counts.loc[key]
        if key in counts.index
        else 0
        for key in sorted(categories.keys())
    }

    return pd.Series(
        data
    )

In [ ]:
categories = {
    1: "zwerfafval",
    3: "afvalberg"
}

In [ ]:
label_file = "../datasets/experiments/zwerfafval/labels/labels_250514_300_v0.1.json"
metadata_file = "../datasets/experiments/zwerfafval/recording_2025-05-14_19-47-40/dataselectie_300.gpkg"

annotations_250514_df = load_and_merge_annotations(label_file, metadata_file, categories)

In [ ]:
label_file = "../datasets/experiments/zwerfafval/labels/labels_260421_1000_v0.1.json"
metadata_file = "../datasets/experiments/zwerfafval/inwinning_260421_selectie_1000_v1.gpkg"

annotations_260421_df = load_and_merge_annotations(label_file, metadata_file, categories)

In [ ]:
annotations_df = pd.concat([annotations_250514_df, annotations_260421_df])

In [ ]:
annotations_df

In [ ]:
annotations_df[annotations_df["category_id"]==1].plot.scatter(x="x_min", y="y_min", ylim=[1, 0], c="area")

In [ ]:
annotations_df[annotations_df["category_id"]==1].plot.scatter(x="x_width", y="y_height")

In [ ]:
annotations_df.sort_values(by="area").tail(10)

In [ ]:
count_categories(annotations_df, categories=categories)

In [ ]:
dataset = annotations_df.groupby(by="name", group_keys=True).apply(func=count_categories, categories=categories)
dataset["group"] = (
    annotations_df
    .groupby(by="name", group_keys=True)
    .apply(lambda df: df["group"].iloc[0])
    .loc[dataset.index]
)

dataset

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.4)
classes = list(categories.values())

X = dataset[["group"]]
y = dataset[classes]

train_ix, not_train_ix = next(splitter.split(X, y, groups=X["group"]))

In [ ]:
print(f"Train percentage: {len(train_ix) / len(dataset)}")

In [ ]:
train_distribution = dataset[classes].iloc[train_ix, :].sum()

print(train_distribution / dataset[classes].sum())

In [ ]:
not_train_data = dataset.iloc[not_train_ix, :]

X = not_train_data[["group"]]
y = not_train_data[classes]

val_ix, test_ix = next(splitter.split(X, y, groups=X["group"]))

In [ ]:
print(f"Validation percentage: {len(val_ix) / len(dataset)}")
print(f"Test percentage: {len(test_ix) / len(dataset)}")

In [ ]:
val_distribution = not_train_data[classes].iloc[val_ix, :].sum()
test_distribution = not_train_data[classes].iloc[test_ix, :].sum()

print(val_distribution / dataset[classes].sum())
print(test_distribution / dataset[classes].sum())

In [ ]:
# TODO

# Write yolo annotations to files in train/val/test folders

# Add empty background fotos

In [ ]:
train_files = dataset.iloc[train_ix, :].index.values
val_files = not_train_data.iloc[val_ix, :].index.values
test_files = not_train_data.iloc[test_ix, :].index.values